# 🧠 Stage 5 — Interpretability, Save & Export
### 🏦 Loan Default Predictor — Home Credit Dataset
---
**Explain predictions with SHAP, save the final model, and export results for Tableau.**

> **Sections 9 & 10**

```
Progress: █████  Stage 5 of 5
```

← [Stage 4](stage_04_models_and_evaluation.ipynb)

---
### 📋 What you will do in this stage:
- Understand **SHAP values** and why model explainability matters in banking
- Build a **SHAP summary plot** — global view of feature impact
- Explain an **individual prediction** with a waterfall chart
- Save the trained model, scaler, and feature list to disk
- Export predictions and feature importances for **Tableau**

⏱️ *Estimated time: 30–45 minutes*

---
> ⚠️ **Prerequisite:** This notebook depends on **Stage 4 (trained `xgb` model, `X_test`, `y_test`, `xgb_proba`)**.  
> Run the previous stage(s) first, **or** run the cell below to reload saved objects.


### ⚙️ Reload Cell
Run this if you are starting fresh without Stage 4 in memory.
It re-runs preprocessing **and** retrains the XGBoost model.


In [ ]:
# ── RELOAD: Preprocessing + model training — run if starting fresh ──
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

df = pd.read_csv("data/raw/application_train.csv")

FEATURES = [
    "EXT_SOURCE_1","EXT_SOURCE_2","EXT_SOURCE_3",
    "AMT_INCOME_TOTAL","AMT_CREDIT","AMT_ANNUITY","AMT_GOODS_PRICE",
    "DAYS_BIRTH","DAYS_EMPLOYED","CODE_GENDER","NAME_EDUCATION_TYPE",
    "NAME_INCOME_TYPE","NAME_FAMILY_STATUS","NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE","FLAG_OWN_CAR","FLAG_OWN_REALTY",
    "CNT_CHILDREN","CNT_FAM_MEMBERS","REGION_RATING_CLIENT",
    "REG_CITY_NOT_WORK_CITY","DEF_30_CNT_SOCIAL_CIRCLE"
]
X = df[FEATURES].copy(); y = df["TARGET"].copy()
X["DAYS_EMPLOYED"] = X["DAYS_EMPLOYED"].replace(365243, 0)
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()
for c in num_cols: X[c] = X[c].fillna(X[c].median())
for c in cat_cols: X[c] = X[c].fillna("Unknown")
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X["CREDIT_INCOME_RATIO"]  = X["AMT_CREDIT"]  / (X["AMT_INCOME_TOTAL"] + 1)
X["ANNUITY_INCOME_RATIO"] = X["AMT_ANNUITY"] / (X["AMT_INCOME_TOTAL"] + 1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
feature_names  = X_train.columns.tolist()

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

scale_ratio = (y_train==0).sum() / (y_train==1).sum()
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05,
                     scale_pos_weight=scale_ratio, eval_metric="auc",
                     random_state=42, n_jobs=-1)
xgb.fit(X_train_res, y_train_res, verbose=False)
xgb_proba = xgb.predict_proba(X_test_scaled)[:, 1]

print(f"✅ Model ready   AUC-ROC: {roc_auc_score(y_test, xgb_proba):.4f}")

---
## Section 9 — Model Interpretability with SHAP

### Why do we need model explanations?

In banking, regulators require that loan decisions be **explainable**.  
A bank can't just say "the AI said no" — it must explain *why*.

**SHAP** (SHapley Additive exPlanations) is a technique from game theory that tells us:
> "How much did each feature **contribute** to this specific prediction?"

> 💡 **SHAP Analogy — Splitting restaurant bills:**  
> Imagine 3 friends order different items at dinner. The bill is $90.  
> SHAP asks: "How much did each person's order **contribute** to the total?"  
> It fairly distributes credit/blame among all features — positive (increases default risk)  
> or negative (decreases default risk).


In [ ]:
# Initialize SHAP explainer — use a sample for speed
# In production you'd use the full test set
SAMPLE_SIZE = 500
X_sample = X_test.iloc[:SAMPLE_SIZE].copy()

explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(scaler.transform(X_sample))

print(f"SHAP values computed for {SAMPLE_SIZE} test samples.")
print(f"Shape of SHAP values: {shap_values.shape}  (samples × features)")

In [ ]:
# SHAP Summary Plot — global view of feature impact
# Each dot = one applicant
# Red = high feature value, Blue = low feature value
# X-axis = SHAP value (impact on model output)

plt.figure(figsize=(11, 8))
shap.summary_plot(
    shap_values,
    X_sample,
    feature_names=feature_names,
    max_display=15,
    show=False
)
plt.title("SHAP Summary Plot — Global Feature Impact", fontsize=14, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("reports/figures/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

#### How to read the SHAP summary plot:
- Each row = one feature
- Each dot = one applicant's SHAP value for that feature
- **Positive SHAP** → increases predicted default probability
- **Negative SHAP** → decreases predicted default probability
- **Red dots** = high feature value, **Blue dots** = low feature value

**Example interpretation:**  
For `EXT_SOURCE_2`: blue dots (low score) cluster on the right (positive SHAP = more default risk).  
High external score → low default risk. ✅ This makes intuitive sense!


In [ ]:
# SHAP Bar Plot — mean absolute impact (simpler view)
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values,
    X_sample,
    feature_names=feature_names,
    plot_type="bar",
    max_display=15,
    show=False
)
plt.title("SHAP Feature Importance (Mean |SHAP|)", fontsize=14, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("reports/figures/shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()

### 9.1 — Individual Prediction Explanation

This is the most powerful part for regulatory compliance:  
we can explain **a single applicant's prediction** — why was their risk high or low?


In [ ]:
# Pick one applicant and explain their prediction
applicant_idx = 5   # Change this to explore different applicants

applicant_data   = scaler.transform(X_sample.iloc[[applicant_idx]])
applicant_pred   = xgb.predict_proba(applicant_data)[0, 1]
applicant_shap   = shap_values[applicant_idx]

print(f"Applicant #{applicant_idx}")
print(f"Predicted default probability: {applicant_pred:.1%}")
print(f"Risk level: {'HIGH' if applicant_pred > 0.5 else 'MEDIUM' if applicant_pred > 0.2 else 'LOW'}")

In [ ]:
# Waterfall plot — shows each feature's contribution for this applicant
shap.waterfall_plot(
    shap.Explanation(
        values=applicant_shap,
        base_values=explainer.expected_value,
        data=X_sample.iloc[applicant_idx].values,
        feature_names=feature_names
    ),
    max_display=12,
    show=False
)
plt.title(f"SHAP Waterfall — Applicant #{applicant_idx}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("reports/figures/shap_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()

#### How to read the waterfall plot:
- **Red bars** → push the prediction *higher* (toward default)
- **Blue bars** → push the prediction *lower* (toward repaid)
- **E[f(X)]** = baseline prediction (average for all applicants)
- **f(X)** = final prediction for this applicant

This is what you'd show in a regulatory audit or loan decision letter.


---
## Section 10 — Save the Model & Export Results

We save everything needed to deploy the model in a Streamlit app or API:
1. The trained model (XGBoost)
2. The scaler (StandardScaler)
3. The feature names list
4. Predictions CSV for Tableau


In [ ]:
import os
os.makedirs("models", exist_ok=True)
os.makedirs("dashboard/data_exports", exist_ok=True)
os.makedirs("reports/figures", exist_ok=True)

In [ ]:
# Save the trained model
joblib.dump(xgb, "models/final_model.pkl")
print("✅ Model saved to models/final_model.pkl")

In [ ]:
# Save the scaler
joblib.dump(scaler, "models/scaler.pkl")
print("✅ Scaler saved to models/scaler.pkl")

In [ ]:
# Save the feature names
joblib.dump(feature_names, "models/feature_names.pkl")
print("✅ Feature names saved to models/feature_names.pkl")

In [ ]:
# Export test predictions for Tableau
predictions_df = pd.DataFrame({
    "SK_ID_CURR"   : df.loc[y_test.index, "SK_ID_CURR"].values,
    "ACTUAL"       : y_test.values,
    "PRED_PROBA"   : xgb_proba,
    "PRED_CLASS"   : (xgb_proba >= 0.5).astype(int),
    "RISK_LEVEL"   : pd.cut(xgb_proba, bins=[0, 0.2, 0.5, 1.0],
                             labels=["Low", "Medium", "High"])
})
predictions_df.to_csv("dashboard/data_exports/predictions.csv", index=False)
print("✅ Predictions exported to dashboard/data_exports/predictions.csv")
print(predictions_df.head())

In [ ]:
# Export feature importances for Tableau
feat_imp_df = pd.DataFrame({
    "feature"    : feature_names,
    "importance" : xgb.feature_importances_
}).sort_values("importance", ascending=False)

feat_imp_df.to_csv("dashboard/data_exports/feature_importance.csv", index=False)
print("✅ Feature importance exported.")
print(feat_imp_df.head(10).to_string(index=False))

In [ ]:
# Export sample applicant data (anonymized) for Tableau Dashboard 3
sample_export = X_test.head(200).copy()
sample_export["ACTUAL"]    = y_test.values[:200]
sample_export["PRED_PROBA"]= xgb_proba[:200]
sample_export.to_csv("dashboard/data_exports/sample_applicants.csv", index=False)
print("✅ Sample applicants exported.")

In [ ]:
# ✅ Final summary
print("=" * 60)
print("  PROJECT COMPLETE — Summary")
print("=" * 60)
print(f"
  Dataset size     : {len(df):,} applicants")
print(f"  Features used    : {len(feature_names)}")
print(f"  Best model       : XGBoost")
print(f"  AUC-ROC (test)   : {roc_auc_score(y_test, xgb_proba):.4f}")
print(f"  Avg Precision    : {average_precision_score(y_test, xgb_proba):.4f}")
print("
  Files saved:")
print("    models/final_model.pkl")
print("    models/scaler.pkl")
print("    models/feature_names.pkl")
print("    dashboard/data_exports/predictions.csv")
print("    dashboard/data_exports/feature_importance.csv")
print("    dashboard/data_exports/sample_applicants.csv")
print("
  Next Step → Open app/app.py and run: streamlit run app/app.py")

---
## 🎉 Project Complete!

Congratulations — you have built a full end-to-end credit risk model!

### 🏆 What you achieved:
- Explored a real-world banking dataset with 307,511 loan applications
- Cleaned and preprocessed messy, imbalanced data
- Engineered new predictive features (credit-to-income ratio, external score averages)
- Trained and compared 3 machine learning models
- Evaluated models rigorously using AUC-ROC, Precision-Recall, and Confusion Matrices
- Explained individual predictions using SHAP values
- Saved a production-ready model and exported results for Tableau

### 🚀 Next Steps:
1. **Run the Streamlit app:** `streamlit run app/app.py`
2. **Build the Tableau dashboard** using the exported CSVs
3. **Findings** on README

